# H1 · Displacement Equation: Solve Weight = Buoyancy in Code
**Ship and Ocean Systems Design 2026 · optional, not graded · about 1 hour**

**Objectives**
1. Write each weight group as a Python function of Δ.
2. Solve Δ = g(Δ) with a loop: Exercise 1.5, Q5, in code.
3. Solve the same equation with Newton–Raphson (Q7).
4. Container ship: find Δ and T last, from the weights (Q8).

Sources: Lecture 1.5 and Exercise 1.5 (slides 12–17); Reading 1.5; Papanikolaou (2026) §2.13.

**How to run:** in Google Colab, first *File › Save a copy in Drive* (so your work is kept), then run the cells from the top with Shift+Enter. Change only the lines marked `# INPUT`.

## 1 · The weights as functions of Δ
The model and data of Exercise 1.5 (Lecture 1.5, slide 12).

In [ ]:
DWT = 37600      # INPUT: deadweight [t]
V = 14.5         # INPUT: service speed [kn]
rho = 1.025      # sea water [t/m3]
CB = 0.757       # Horn, Exercise 1.4
LB = 5.92        # L/B, Exercise 1.4
BT = 2.69        # B/T
DT = 1.36        # D/T
K = 0.032        # Watson steel coefficient (tankers)
c_o = 0.025      # outfit = c_o * Delta
k_m = 0.10       # machinery [t per kW]
C_adm = 450      # Admiralty constant

def main_dimensions(delta):
    L = (delta * LB**2 * BT / (rho * CB)) ** (1 / 3)
    B = L / LB
    T = B / BT
    D = DT * T
    return L, B, T, D

def steel_weight(delta):
    L, B, T, D = main_dimensions(delta)
    E = L * (B + T) + 0.85 * L * (D - T)     # Watson's E number
    return K * E**1.36

def outfit_weight(delta):
    return c_o * delta

def machinery_weight(delta):
    power = delta**(2 / 3) * V**3 / C_adm    # kW, Admiralty formula
    return k_m * power

def g(delta):
    """Total weight of a ship of displacement delta."""
    return DWT + steel_weight(delta) + outfit_weight(delta) + machinery_weight(delta)

print(f"g(DWT) = {g(DWT):,.0f} t")

## 2 · Fixed-point iteration (Q5 in code)
Start at Δ = DWT and repeat Δ ← g(Δ) until the change is below 1 t.
*Run section 1 first: this section uses its inputs and functions.*

In [ ]:
delta = DWT
change = 1e9
step = 0
while change >= 1.0:
    new_delta = g(delta)
    change = abs(new_delta - delta)
    delta = new_delta
    step = step + 1
    print(f"step {step}: Delta = {delta:,.1f} t, change = {change:,.1f} t")

L, B, T, D = main_dimensions(delta)
print(f"L = {L:.1f} m, B = {B:.2f} m, T = {T:.2f} m, D = {D:.2f} m")
print(f"steel {steel_weight(delta):,.0f} t, outfit {outfit_weight(delta):,.0f} t, machinery {machinery_weight(delta):,.0f} t")

Compare with your hand solution of Q5 (Lecture 1.5, slides 13–17).

## 3 · Newton–Raphson (Q7)
Write f(Δ) = Δ − g(Δ), zero at equilibrium, and follow the tangent: Δ ← Δ − f(Δ)/f′(Δ).
The slope f′ is taken numerically: the change of f over 1 t.
*Run section 1 first: this section uses its inputs and functions.*

In [ ]:
def f(delta):
    return delta - g(delta)

delta = DWT
for step in range(1, 10):
    slope = f(delta + 1) - f(delta)          # f'(Delta), numerically
    new_delta = delta - f(delta) / slope
    change = abs(new_delta - delta)
    delta = new_delta
    print(f"step {step}: Delta = {delta:,.1f} t, change = {change:,.3f} t")
    if change < 1.0:
        break

## 4 · Container ship: Δ and T come last (Q8)
For a volume carrier the hull comes from the cargo; the weights then give Δ, and the draught follows from buoyancy (Lecture 1.5, slide 5): T = Δ / (ρ·C_B·L·B).
Example: the 2,000 TEU ship of Lecture 1.4 and of H0, with the container fits of Appendix A.3.
This section runs on its own: it has its own inputs.

In [ ]:
import math

rho = 1.025                       # sea water [t/m3]
TEU = 2000                        # INPUT: number of containers
cargo = 10 * TEU                  # 10 t per TEU, as in the project brief
DWT_c = cargo / 0.75              # Fig. A.49: cargo = 0.75 DWT

L_c = 3.54132 * DWT_c**0.388442          # Fig. A.31
B_c = 1.55219 * DWT_c**0.284381          # Fig. A.32
CB_c = 0.908683 * DWT_c**-0.0320423      # Fig. A.38
lightship = 0.973087 * DWT_c**0.91242    # Fig. A.36

delta_c = lightship + DWT_c                       # weights first
T_c = delta_c / (rho * CB_c * L_c * B_c)          # draught last
T_fit = -17.1581 + 2.72338 * math.log(DWT_c)      # Fig. A.34, for comparison

print(f"DWT = {DWT_c:,.0f} t, lightship = {lightship:,.0f} t, Delta = {delta_c:,.0f} t")
print(f"L = {L_c:.1f} m, B = {B_c:.2f} m, CB = {CB_c:.3f}")
print(f"T from buoyancy = {T_c:.2f} m, T from the fit = {T_fit:.2f} m")

## Questions · answer in one or two lines each
**Q7** How many steps did fixed-point iteration need, and how many Newton–Raphson? Why is Newton faster?

**Q8** In section 4, does T from buoyancy agree with the fit? If a port allowed only 9.5 m, which dimension would you change, and why?

*Your answers:*
